# Normalización y Lematización de un libro completo

**Checkpoint 3 — Procesamiento de Lenguaje Natural (Hybridge)**

Pipeline de limpieza aplicado a un libro en `.txt`:

1. Carga robusta del texto (codificación y caracteres rotos)
2. Limpieza básica con expresiones regulares
3. Tokenización con spaCy
4. Eliminación de ruido (stopwords y puntuación)
5. Lematización y normalización a minúsculas
6. Comparativa Stemming (NLTK) vs Lematización (spaCy)
7. Exportación del corpus limpio

## 0. Configuración

In [1]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy
from nltk.stem import SnowballStemmer

# Book file and output folder (relative to the project root)
RUTA_LIBRO = Path("libro.txt")
CARPETA_SALIDA = Path("salida")
CARPETA_SALIDA.mkdir(exist_ok=True)

# Load the Spanish model; NER is not needed for cleaning, so we disable it for speed
nlp = spacy.load("es_core_news_sm", disable=["ner"])
nlp.max_length = 3_000_000  # allow long texts (a whole book)

print(f"spaCy {spacy.__version__} | pipeline: {nlp.pipe_names}")

spaCy 3.8.16 | pipeline: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer']


## 1. Carga del texto

Los `.txt` descargados de internet suelen traer problemas de codificación: acentos que aparecen como `Ã­` o `Ã±` (*mojibake*). Esta función prueba la codificación correcta y repara ese caso antes de procesar nada, porque un acento roto hace que spaCy lematice mal la palabra.

In [2]:
def cargar_texto(ruta: Path) -> str:
    """Read a text file trying common encodings and repair double-encoded UTF-8."""
    raw = ruta.read_bytes()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            texto = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue

    # Repair mojibake such as 'tenÃ\xada' -> 'tenía' (UTF-8 bytes read as Latin-1)
    if re.search(r"Ã[\x80-\xbf¡-¿]", texto):
        try:
            texto = texto.encode("cp1252", errors="strict").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            texto = texto.encode("latin-1", errors="ignore").decode("utf-8", errors="ignore")
    return texto


texto_original = cargar_texto(RUTA_LIBRO)
print(f"Caracteres: {len(texto_original):,}")
print(texto_original[:500])

Caracteres: 458,735
The Project Gutenberg eBook of El sombrero de tres picos
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using


## 2. Limpieza básica (regex)

Antes de tokenizar quitamos lo que no es parte de la obra:

- Encabezado y licencia de Project Gutenberg (si el libro viene de ahí)
- Saltos de línea que parten las oraciones a la mitad
- Espacios repetidos y comillas tipográficas inconsistentes
- Introducción, notas y vocabulario en inglés del editor (es una edición escolar de 1907).

In [3]:
def quitar_gutenberg(texto: str) -> str:
    """Keep only the body of a Project Gutenberg book (no-op for other sources)."""
    inicio = re.search(r"\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", texto, re.I)
    fin = re.search(r"\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG", texto, re.I)
    desde = inicio.end() if inicio else 0
    hasta = fin.start() if fin else len(texto)
    return texto[desde:hasta]

# Section headings of the editor's English apparatus that follow the novel
PATRON_APENDICES = re.compile(r"^\s*(NOTES|VOCABULARY|FOOTNOTES)\s*$", re.M)


def recortar_novela(texto: str) -> str:
    """Keep only Alarcón's text: from his dedication up to the editor's notes/vocabulary."""
    inicio = re.search(r"^\s*AL SEÑOR\s*$", texto, re.M)
    desde = inicio.start() if inicio else 0
    # Search after the dedication so the table-of-contents entries are skipped
    fin = PATRON_APENDICES.search(texto, desde)
    hasta = fin.start() if fin else len(texto)
    print(f"Recorte: caracteres {desde:,} a {hasta:,} de {len(texto):,}")
    return texto[desde:hasta]


# Paragraphs added by Project Gutenberg volunteers (credits, transcriber notes)
PATRON_CREDITOS = re.compile(
    r"^(produced by|e-?text prepared by|transcriber'?s? notes?|nota del transcriptor"
    r"|end of (the )?project gutenberg)",
    re.I,
)


def limpiar_texto(texto: str) -> list[str]:
    """Normalize whitespace/quotes and return the text split into paragraphs."""
    texto = quitar_gutenberg(texto)
    texto = recortar_novela(texto)
    texto = texto.replace("\r\n", "\n")
    texto = re.sub(r"[“”«»]", '"', texto)
    texto = re.sub(r"[‘’]", "'", texto)
    texto = re.sub(r"_+", "", texto)  # Gutenberg italics markers

    # A blank line separates paragraphs; single line breaks are just wrapping
    parrafos = re.split(r"\n\s*\n", texto)
    parrafos = [re.sub(r"\s+", " ", p).strip() for p in parrafos]
    return [p for p in parrafos if len(p) > 1 and not PATRON_CREDITOS.match(p)]

parrafos = limpiar_texto(texto_original)
print(f"Párrafos: {len(parrafos):,}")
for p in parrafos[:3]:
    print("-", p[:150])

print("Último párrafo:", parrafos[-1][:200])

Recorte: caracteres 19,938 a 174,582 de 438,901
Párrafos: 940
- AL SEÑOR
- D. JOSÉ SALVADOR DE SALVADOR
- Dedico esta obra.
Último párrafo: FIN.


## 3. Tokenización

Procesamos el libro párrafo por párrafo con `nlp.pipe`. Es más rápido que pasar todo el texto de golpe y no satura la memoria con libros largos.

In [4]:
docs = list(nlp.pipe(parrafos, batch_size=64))
tokens_totales = [token for doc in docs for token in doc]

print(f"--- Tokenización (Total tokens: {len(tokens_totales):,}) ---")
print([t.text for t in tokens_totales[:20]])

--- Tokenización (Total tokens: 30,395) ---
['AL', 'SEÑOR', 'D.', 'JOSÉ', 'SALVADOR', 'DE', 'SALVADOR', 'Dedico', 'esta', 'obra', '.', 'P.', 'A.', 'DE', 'ALARCÓN', '.', 'JULIO', 'de', '1874', '.']


## 4. Eliminación de ruido (stopwords y puntuación)

In [5]:
def es_relevante(token) -> bool:
    """Keep only alphabetic content words (drops stopwords, punctuation, numbers, spaces)."""
    return token.is_alpha and not token.is_stop

tokens_relevantes = [t for t in tokens_totales if es_relevante(t)]
tokens_ruido = [t for t in tokens_totales if t.is_stop]

print("--- Filtrado de Stop Words ---")
print("Palabras eliminadas (ruido):", [t.text for t in tokens_ruido[:10]], "...")
print("Palabras conservadas (contenido):", [t.text for t in tokens_relevantes[:10]], "...")
reduccion = 100 * (1 - len(tokens_relevantes) / len(tokens_totales))
print(f"Reducción: de {len(tokens_totales):,} a {len(tokens_relevantes):,} tokens ({reduccion:.1f}% menos)")

--- Filtrado de Stop Words ---
Palabras eliminadas (ruido): ['AL', 'DE', 'esta', 'DE', 'de', 'DEL', 'Pocos', 'aun', 'a', 'los'] ...
Palabras conservadas (contenido): ['SEÑOR', 'JOSÉ', 'SALVADOR', 'SALVADOR', 'Dedico', 'obra', 'ALARCÓN', 'JULIO', 'PREFACIO', 'AUTOR'] ...
Reducción: de 30,395 a 9,893 tokens (67.5% menos)


## 5. Lematización y normalización

Cada palabra se reduce a su lema y se pasa a minúsculas, para que *hablo*, *hablaré* y *habla* cuenten como el mismo concepto: *hablar*.

In [6]:
tokens_normalizados = [t.lemma_.lower() for t in tokens_relevantes]

# Words whose lemma differs from the lowercase form (most informative examples)
cambios = Counter(
    (t.text.lower(), t.lemma_.lower())
    for t in tokens_relevantes
    if t.text.lower() != t.lemma_.lower()
)

print("--- Lematización y Normalización ---")
print(f"Tokens procesados: {len(tokens_normalizados):,}")
print(f"Vocabulario antes: {len({t.text.lower() for t in tokens_relevantes}):,} formas distintas")
print(f"Vocabulario después: {len(set(tokens_normalizados)):,} lemas distintos")
print("\nTransformaciones más frecuentes (palabra → lema):")
for (original, lema), n in cambios.most_common(15):
    print(f"  {original:>15} → {lema:<15} ({n} veces)")

--- Lematización y Normalización ---
Tokens procesados: 9,893
Vocabulario antes: 4,656 formas distintas
Vocabulario después: 3,702 lemas distintos

Transformaciones más frecuentes (palabra → lema):
          exclamó → exclamar        (28 veces)
             ojos → ojo             (22 veces)
          hubiera → haber           (16 veces)
              iba → ir              (16 veces)
             años → año             (15 veces)
            manos → mano            (15 veces)
             digo → decir           (15 veces)
            viene → venir           (14 veces)
            picos → pico            (13 veces)
          parecía → parecer         (13 veces)
        respondió → responder       (13 veces)
            horas → hora            (13 veces)
         contestó → contestar       (12 veces)
            media → medio           (12 veces)
            cosas → cosa            (11 veces)


## 6. Stemming vs Lematización

El **stemming** (NLTK) corta sufijos con reglas fijas, sin entender la palabra. La **lematización** (spaCy) usa el contexto y un diccionario para dar la forma real. Comparamos ambas sobre los verbos y sustantivos más frecuentes del libro.

In [7]:
stemmer = SnowballStemmer("spanish")

# Pick the most frequent verbs and nouns from the book itself, so the table is never empty
frecuentes = Counter(
    t.text.lower() for t in tokens_relevantes if t.pos_ in ("VERB", "NOUN")
    and t.text.lower() != t.lemma_.lower()
)
ejemplos = {}
for t in tokens_relevantes:
    palabra = t.text.lower()
    if palabra in dict(frecuentes.most_common(15)) and palabra not in ejemplos:
        ejemplos[palabra] = t.lemma_.lower()

df = pd.DataFrame(
    [
        {
            "Original": palabra,
            "Stemming (corte)": stemmer.stem(palabra),
            "Lematización (diccionario)": lema,
            "¿Coinciden?": stemmer.stem(palabra) == lema,
        }
        for palabra, lema in ejemplos.items()
    ]
)
print(df.to_string(index=False))
print(f"\nEl stem coincide con el lema en {df['¿Coinciden?'].mean():.0%} de los casos.")

 Original Stemming (corte) Lematización (diccionario)  ¿Coinciden?
     años             años                        año        False
    picos              pic                       pico        False
 contestó          contest                  contestar        False
    cosas              cos                       cosa        False
 continuó          continu                  continuar        False
    manos              man                       mano        False
    viene             vien                      venir        False
     digo              dig                      decir        False
    hacía              hac                      hacer        False
     uvas             uvas                        uva        False
    horas              hor                      horas        False
     ojos             ojos                        ojo        False
  parecía            parec                    parecer        False
  exclamó           exclam                   exclamar        F

## 7. Lemas más frecuentes

In [8]:
frecuencias = Counter(tokens_normalizados).most_common(20)
df_frec = pd.DataFrame(frecuencias, columns=["Lema", "Frecuencia"])
print(df_frec.to_string(index=False))

      Lema  Frecuencia
     lucas         156
corregidor         155
       tío         140
 frasquita         123
      señá         103
     señor          92
     mujer          62
    molino          58
     decir          54
   garduña          53
       ver          51
     haber          50
    hombre          49
      casa          44
     noche          43
    tiempo          40
    ciudad          40
  alguacil          38
    puerta          37
     venir          36


## 8. Corpus limpio y exportación

Armamos un documento lematizado por párrafo (listo para `CountVectorizer` o `TfidfVectorizer` en la siguiente etapa) y lo guardamos en `salida/`.

In [9]:
corpus_lematizado = []  # one cleaned document per paragraph

for doc in docs:
    lemas = [t.lemma_.lower() for t in doc if es_relevante(t)]
    if lemas:
        corpus_lematizado.append(" ".join(lemas))

ruta_corpus = CARPETA_SALIDA / "corpus_lematizado.txt"
ruta_corpus.write_text("\n".join(corpus_lematizado), encoding="utf-8")
df_frec.to_csv(CARPETA_SALIDA / "frecuencias.csv", index=False, encoding="utf-8")

print(f"Documentos en el corpus: {len(corpus_lematizado):,}")
print(f"Guardado en: {ruta_corpus}")
print("\nEjemplo:", corpus_lematizado[0][:300])

Documentos en el corpus: 909
Guardado en: salida/corpus_lematizado.txt

Ejemplo: señor


## Conclusiones

- **Conocer la fuente es parte de la limpieza.** El archivo era una edición escolar de 1907 con prefacio, introducción, notas y vocabulario en inglés. Solo un tercio del texto era la novela. Sin recortarlo, las palabras más frecuentes eran *to*, *the* y *of*, y spaCy inventaba lemas como *one → onir* o *name → namir*.
- **Las stopwords son la mayor parte del texto.** Al quitar stopwords y puntuación, los tokens bajaron de 30,395 a 9,893 (67.5% menos).
- **La lematización compacta el vocabulario.** Las 4,656 formas distintas quedaron en 3,702 lemas, porque conjugaciones y plurales se agrupan (*exclamó → exclamar*, *ojos → ojo*).
- **Stemming y lematización casi nunca coinciden.** El stemming produce raíces que no son palabras (*respond*, *exclam*, *dig*), mientras que la lematización devuelve palabras reales (*responder*, *exclamar*, *decir*).
- **El resultado refleja la obra.** Los lemas más frecuentes son los personajes y lugares de la novela: Lucas, el Corregidor, la señá Frasquita, el molino y Garduña.